# Fase 2: Módulo de Visión y Data Augmentation (OpenCV + YOLOv8-cls)
## Sistema de Triaje Multimodal Clínico (NLP + Visión por Computadora)

Este notebook implementa el pipeline de Visión por Computadora:
1. **Configuración del entorno e instalación de Ultralytics YOLOv8**.
2. **Entrenamiento de YOLOv8n-cls** sobre el dataset clínico generado (4 clases, 35 épocas, `imgsz=224`, `batch=16`).
3. **Evaluación de métricas** (Top-1 Accuracy, Matriz de Confusión y Confianza promedio por clase).
4. **Exportación del mejor peso (`best.pt`)** hacia `backend/models/best.pt`.
5. **Pruebas de inferencia visual** sobre 4 casos de prueba (1 por cada clase clínica).

In [ ]:
# Instalación de dependencias (para Google Colab)
!pip install --quiet ultralytics opencv-python-headless pillow

In [ ]:
import os
import shutil
import json
import glob
import numpy as np
from PIL import Image
from ultralytics import YOLO

print("Ultralytics y librerías inicializadas correctamente.")

### 1. Ubicación y Verificación del Dataset Clínico

In [ ]:
dataset_dir = "dataset_clinico"
if not os.path.exists(dataset_dir):
    dataset_dir = "training/dataset_clinico"

train_dir = os.path.join(dataset_dir, "train")
val_dir = os.path.join(dataset_dir, "val")

classes = sorted([d for d in os.listdir(val_dir) if os.path.isdir(os.path.join(val_dir, d))])
print(f"Clases detectadas ({len(classes)}):")
for c in classes:
    n_tr = len(glob.glob(os.path.join(train_dir, c, '*.jpg')))
    n_va = len(glob.glob(os.path.join(val_dir, c, '*.jpg')))
    print(f"  - {c}: {n_tr} train | {n_va} val")

### 2. Entrenamiento de YOLOv8n-cls (35 Épocas, imgsz=224, batch=16)

In [ ]:
model = YOLO("yolov8n-cls.pt")

results = model.train(
    data=dataset_dir,
    epochs=35,
    imgsz=224,
    batch=16,
    seed=42,
    project="runs/classify",
    name="triaje_yolo_cls",
    exist_ok=True,
    verbose=True
)

print("Entrenamiento completado.")

### 3. Evaluación de Métricas de Validación

In [ ]:
best_pt = os.path.join(model.trainer.save_dir, "weights", "best.pt")
print(f"Mejor peso: {best_pt}")

eval_model = YOLO(best_pt)
val_metrics = eval_model.val()

print(f"Top-1 Accuracy en Validación: {val_metrics.top1 * 100:.2f}%")
print(f"Top-5 Accuracy en Validación: {val_metrics.top5 * 100:.2f}%")

### 4. Exportación del Peso hacia `backend/models/best.pt`

In [ ]:
target_backend_pt = "../backend/models/best.pt"
if os.path.exists("../backend/models"):
    shutil.copy2(best_pt, target_backend_pt)
    print(f"Copiado exitosamente a: {target_backend_pt}")
else:
    print(f"Peso disponible en: {best_pt}")

### 5. Pruebas de Inferencia Visual sobre Casos de Validación

In [ ]:
for cls_name in classes:
    sample_imgs = glob.glob(os.path.join(val_dir, cls_name, "*.jpg"))
    if sample_imgs:
        test_p = sample_imgs[0]
        pred = eval_model.predict(source=test_p, imgsz=224, verbose=False)[0]
        top1_name = pred.names[pred.probs.top1]
        top1_conf = float(pred.probs.top1conf)
        print(f"Clase Esperada: {cls_name}")
        print(f"  -> Predicción: {top1_name} (Confianza: {top1_conf * 100:.2f}%)\n")